# Initialization and Optimizers: Where to Start, and How to Step

The full code of [Initialization and Optimizers: Where to Start, and How to Step](https://residual-stream.lvtan71.workers.dev/posts/initialization-and-optimizers/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/initialization-and-optimizers) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/initialization-and-optimizers.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "initialization-and-optimizers"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `depth_signal.py`

Activation and gradient size through 20 layers of width 256 for five initializations, median of five seeds.

In [ ]:
!{sys.executable} depth_signal.py

The post printed:

```text
                          activation size after layer         gradient reaching 
                            1         5        10        20   layer 20   layer 1
N(0, 0.01^2), ReLU   1.13e-01  1.98e-05  3.68e-10  1.37e-19   1.14e-01  1.21e-19
N(0, 1), ReLU        1.13e+01  1.98e+05  3.68e+10  1.37e+21   1.14e+01  1.21e+21
Xavier 1/n, ReLU     7.07e-01  1.89e-01  3.35e-02  1.14e-03   7.15e-01  1.00e-03
He 2/n, ReLU         1.00e+00  1.07e+00  1.07e+00  1.16e+00   1.01e+00  1.03e+00
Xavier 1/n, tanh     6.28e-01  3.21e-01  2.27e-01  1.58e-01   9.73e-01  1.88e-01
```

## `deep_init.py`

The four-way router with ten hidden ReLU layers, trained from three initializations with SGD and Adam: first-layer gradient and loss at 0, 100 and 1,000 steps. Its `init` is imported by `adam_eps.py`.

In [ ]:
!{sys.executable} deep_init.py

The post printed:

```text
               optimizer    largest first-layer gradient   train loss at 0 / 100 / 1,000 steps
N(0, 0.01^2)   SGD 0.1                           7.3e-15          1.386   1.387   1.387
N(0, 0.01^2)   Adam 0.001                        7.3e-15          1.386   1.386   1.386
Xavier 1/n     SGD 0.1                           6.9e-04          1.386   1.376   0.001
Xavier 1/n     Adam 0.001                        6.9e-04          1.386   0.091   0.000
He 2/n         SGD 0.1                           2.5e-02          1.399   0.700   0.000
He 2/n         Adam 0.001                        2.5e-02          1.399   0.011   0.000
```

## `momentum.py`

Plain gradient descent against heavy-ball momentum on the tokens-against-words fit of Learning Is Optimization: steps to converge, against theory.

In [ ]:
!{sys.executable} momentum.py

The post printed:

```text
eigenvalues 0.2407 and 30.72, kappa 128
gradient descent, eta 0.0646:              882 steps   (theory: error shrinks by 0.9844 per step)
momentum, eta 0.1099, beta 0.701:     108 steps   (theory: 0.8374 per step)
momentum, beta 0.5, same effective step:     736 steps
momentum, beta 0.9, same effective step:     630 steps
momentum, beta 0.99, same effective step:    2302 steps
first t with sqrt(beta)^t < 1e-6: 78; with t * sqrt(beta)^t < 1e-6: 104
```

## `bias_correction.py`

Adam's moving averages on a constant gradient, with and without bias correction, and the first step for gradients of very different sizes.

In [ ]:
!{sys.executable} bias_correction.py

The post printed:

```text
t=1: m 1.00e-04  v 1.00e-09   step without correction 0.03162   with correction 0.01000
t=2: m 1.90e-04  v 2.00e-09   step without correction 0.04250   with correction 0.01000
t=3: m 2.71e-04  v 3.00e-09   step without correction 0.04950   with correction 0.01000
t=4: m 3.44e-04  v 3.99e-09   step without correction 0.05442   with correction 0.01000
t=5: m 4.10e-04  v 4.99e-09   step without correction 0.05797   with correction 0.01000
gradient   1e-06: first Adam step 0.0100   first gradient-descent step 1.0e-08
gradient    0.01: first Adam step 0.0100   first gradient-descent step 1.0e-04
gradient      10: first Adam step 0.0100   first gradient-descent step 1.0e-01
uncorrected / corrected: peak 6.57 at t=12, 3.24 at t=100, 1.26 at t=1000
```

## `adam_eps.py`

The deep router from the tiny start, trained by Adam with epsilon 1e-8 and 1e-30, five seeds.

In [ ]:
!{sys.executable} adam_eps.py

The post printed:

```text
Adam eps 1e-08: train loss after 1,000 steps, 5 seeds: 1.386 1.386 1.386 1.386 1.386
Adam eps 1e-30: train loss after 1,000 steps, 5 seeds: 0.169 1.386 1.386 0.067 1.386
```

## `race_both.py`

Runs `race.py`, then `race.py length`.

In [ ]:
!{sys.executable} race_both.py

The post printed:

```text
bag of words:
gradient descent  after   30 steps 0.3052 (eta 10)   after  300 steps 0.3011 (eta 10)   after 3000 steps 0.3011 (eta 3.2)
momentum 0.9      after   30 steps 0.3105 (eta 3.2)   after  300 steps 0.3011 (eta 10)   after 3000 steps 0.3011 (eta 0.32)
Adam              after   30 steps 0.3158 (eta 0.1)   after  300 steps 0.3011 (eta 0.1)   after 3000 steps 0.3011 (eta 0.01)
plus raw question length:
gradient descent  after   30 steps 1.3734 (eta 0.0032)   after  300 steps 1.3021 (eta 0.0032)   after 3000 steps 0.8829 (eta 0.0032)
momentum 0.9      after   30 steps 1.3237 (eta 0.0032)   after  300 steps 0.8877 (eta 0.0032)   after 3000 steps 0.2948 (eta 0.01)
Adam              after   30 steps 0.3495 (eta 0.1)   after  300 steps 0.2917 (eta 0.32)   after 3000 steps 0.2914 (eta 0.01)
```

## `curvature.py`

The largest curvature of the race's objective at the start, without and with the length feature, and the step size it allows.

In [ ]:
!{sys.executable} curvature.py

The post printed:

```text
bag of words     largest curvature     0.51, so gradient descent is stable only below 2 / that = 3.9118
plus raw length  largest curvature   543.91, so gradient descent is stable only below 2 / that = 0.0037
```

## `race_curves.py`

The objective curves behind the post's chart: each optimizer at its best learning rate, median of five splits.

In [ ]:
!{sys.executable} race_curves.py

The post printed:

```text
gradient descent: 1.3833 1.3821 1.3815 1.3804 1.3792 1.3769 1.3729 1.3661 1.3541 1.3333 1.2981 1.2400 1.1496 1.0234 0.8678
momentum 0.9, same eta: 1.3833 1.3863 1.3809 1.3796 1.3707 1.3541 1.3187 1.2596 1.1643 1.0320 0.8725 0.7008 0.5382 0.4079 0.3314
momentum 0.9, eta 0.01: 1.4433 1.7060 1.8667 6.4745 13.3828 12.4024 21.6123 14.2705 13.1169 6.2789 6.9338 0.4115 0.3242 0.3016 0.2945
Adam: 1.3531 1.2584 1.1951 1.0186 0.8411 0.5947 0.4377 0.3681 0.3287 0.3085 0.2982 0.2935 0.2917 0.2914 0.2914
steps: 1 2 3 6 10 18 32 56 100 178 316 562 1000 1778 3162
```

## `warmup_noise.py`

Adam on pure-noise gradients for 100,000 parameters: the typical step and the share of runaway per-parameter rates, by step.

In [ ]:
!{sys.executable} warmup_noise.py

The post printed:

```text
step     1: typical update 1.000 x eta;  parameters with a rate over 3x its true value: 25.97%
step     2: typical update 0.708 x eta;  parameters with a rate over 3x its true value: 10.42%
step     5: typical update 0.452 x eta;  parameters with a rate over 3x its true value:  0.97%
step    10: typical update 0.329 x eta;  parameters with a rate over 3x its true value:  0.02%
step   100: typical update 0.231 x eta;  parameters with a rate over 3x its true value:  0.00%
step  1000: typical update 0.229 x eta;  parameters with a rate over 3x its true value:  0.00%
step  5000: typical update 0.229 x eta;  parameters with a rate over 3x its true value:  0.00%
long-run update size, sqrt((1 - b1) / (1 + b1)) = 0.229
```

## `decay.py`

No decay, SGD with L2, Adam with L2 and AdamW on the router, mini-batches of 16, 20 splits: validation choice, weight sizes, test results. The argument sets the steps per run (default 3,000).

In [ ]:
!{sys.executable} decay.py

The post printed:

```text
3,000 steps, mini-batches of 16
no decay  λ 0      val CE 0.881   weight size: common words 0.852, rare words 0.927, ratio 0.92
SGD + L2  λ 0.0003 val CE 0.914   weight size: common words 0.769, rare words 0.410, ratio 1.88
SGD + L2  λ 0.001  val CE 0.888   weight size: common words 0.626, rare words 0.328, ratio 1.91
SGD + L2  λ 0.003  val CE 0.893   weight size: common words 0.496, rare words 0.245, ratio 2.02
Adam + L2 λ 0.0001 val CE 0.912   weight size: common words 0.818, rare words 0.523, ratio 1.57
Adam + L2 λ 0.0003 val CE 0.907   weight size: common words 0.747, rare words 0.428, ratio 1.75
Adam + L2 λ 0.001  val CE 0.889   weight size: common words 0.624, rare words 0.333, ratio 1.88
Adam + L2 λ 0.003  val CE 0.894   weight size: common words 0.497, rare words 0.249, ratio 2.00
AdamW     λ 0.1    val CE 0.822   weight size: common words 0.613, rare words 0.664, ratio 0.92
AdamW     λ 0.3    val CE 0.819   weight size: common words 0.502, rare words 0.525, ratio 0.96
AdamW     λ 1      val CE 0.894   weight size: common words 0.307, rare words 0.312, ratio 0.98
chosen on validation: SGD + L2 λ 0.001, Adam + L2 λ 0.001, AdamW λ 0.3
on test, at the chosen λ:  no decay CE 0.870 acc 0.691, SGD + L2 CE 0.896 acc 0.640, Adam + L2 CE 0.897 acc 0.631, AdamW CE 0.822 acc 0.688
test, AdamW minus SGD + L2: CE -0.074 ± 0.042, accuracy +0.047 ± 0.040
test, AdamW minus no decay: CE -0.048 ± 0.047, accuracy -0.004 ± 0.017
test, SGD + L2 minus no decay: CE +0.026 ± 0.041, accuracy -0.051 ± 0.034
```

## `decay.py 12000`

No decay, SGD with L2, Adam with L2 and AdamW on the router, mini-batches of 16, 20 splits: validation choice, weight sizes, test results. The argument sets the steps per run (default 3,000).

In [ ]:
!{sys.executable} decay.py 12000

The post printed:

```text
12,000 steps, mini-batches of 16
no decay  λ 0      val CE 1.128   weight size: common words 1.342, rare words 1.511, ratio 0.89
SGD + L2  λ 0.0003 val CE 0.912   weight size: common words 0.768, rare words 0.423, ratio 1.82
SGD + L2  λ 0.001  val CE 0.887   weight size: common words 0.624, rare words 0.329, ratio 1.90
SGD + L2  λ 0.003  val CE 0.892   weight size: common words 0.496, rare words 0.246, ratio 2.02
Adam + L2 λ 0.0001 val CE 0.958   weight size: common words 0.904, rare words 0.516, ratio 1.76
Adam + L2 λ 0.0003 val CE 0.916   weight size: common words 0.771, rare words 0.428, ratio 1.81
Adam + L2 λ 0.001  val CE 0.890   weight size: common words 0.628, rare words 0.333, ratio 1.89
Adam + L2 λ 0.003  val CE 0.894   weight size: common words 0.498, rare words 0.249, ratio 2.01
AdamW     λ 0.1    val CE 0.966   weight size: common words 1.025, rare words 1.093, ratio 0.94
AdamW     λ 0.3    val CE 0.855   weight size: common words 0.764, rare words 0.818, ratio 0.93
AdamW     λ 1      val CE 0.867   weight size: common words 0.340, rare words 0.361, ratio 0.94
chosen on validation: SGD + L2 λ 0.001, Adam + L2 λ 0.001, AdamW λ 0.3
on test, at the chosen λ:  no decay CE 1.098 acc 0.699, SGD + L2 CE 0.894 acc 0.634, Adam + L2 CE 0.892 acc 0.630, AdamW CE 0.856 acc 0.676
test, AdamW minus SGD + L2: CE -0.037 ± 0.060, accuracy +0.042 ± 0.040
test, AdamW minus no decay: CE -0.242 ± 0.098, accuracy -0.022 ± 0.029
test, SGD + L2 minus no decay: CE -0.205 ± 0.095, accuracy -0.065 ± 0.035
```

## `adam_memory.py`

Bytes of training state per parameter for an 8B model under mixed precision.

In [ ]:
!{sys.executable} adam_memory.py

The post printed:

```text
BF16 weights          2 bytes/param     16 GB
BF16 gradients        2 bytes/param     16 GB
FP32 master weights   4 bytes/param     32 GB
Adam m (FP32)         4 bytes/param     32 GB
Adam v (FP32)         4 bytes/param     32 GB
total                16 bytes/param    128 GB, 1.6 H100s of memory before a single activation
```